In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time

In [2]:
#REQUIRED FOR ALL
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/_NIFTY_IDX__202507041318.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

# Save and reload to Parquet (optional)
try:

    print("spot_df loaded successfully from parquet.")
except Exception as e:
    print(f"Error saving/loading spot_df: {e}")


spot_df preview:
             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20
spot_df loaded successfully from parquet.


In [3]:
#REQUIRED FOR ALL
#load spot data

# --- 2. Ensure Datetime is datetime and sort for time-based ops ---
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# --- 3. Set Datetime as index for resampling ---
spot_df = spot_df.set_index('Datetime')

# --- 4. Filter to regular NIFTY trading hours (avoid pre/post-market ticks) ---
spot_df = spot_df.between_time('09:15:00', '15:25:00')

# --- 5. Resample to 60-min OHLCV bars. Offset=15min for NIFTY standard (9:15 open) ---
spot_1min = spot_df.resample(
    '1min', 
    origin='start_day', 
    offset='15min', 
    label='left', 
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_1min = spot_1min.reset_index()


# --- 9. Preview final 60-min OHLC + EMA DataFrame ---
print(spot_1min.tail())

# --- 5. Resample to 60-min OHLCV bars. Offset=15min for NIFTY standard (9:15 open) ---
spot_5min = spot_df.resample(
    '5min', 
    origin='start_day', 
    offset='15min', 
    label='left', 
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_5min = spot_5min.reset_index()


# --- 9. Preview final 60-min OHLC + EMA DataFrame ---
print(spot_5min.tail())


                  Datetime      Open      High       Low     Close
369690 2025-06-13 15:21:00  24720.50  24722.45  24718.20  24721.40
369691 2025-06-13 15:22:00  24721.05  24728.85  24720.75  24727.45
369692 2025-06-13 15:23:00  24726.70  24728.90  24725.95  24728.10
369693 2025-06-13 15:24:00  24728.15  24733.15  24727.40  24729.35
369694 2025-06-13 15:25:00  24728.55  24734.40  24728.45  24733.55
                 Datetime      Open      High       Low     Close
74740 2025-06-13 15:05:00  24702.60  24727.90  24702.60  24727.90
74741 2025-06-13 15:10:00  24727.95  24728.45  24717.20  24725.60
74742 2025-06-13 15:15:00  24725.15  24725.90  24709.75  24717.95
74743 2025-06-13 15:20:00  24718.70  24733.15  24718.10  24729.35
74744 2025-06-13 15:25:00  24728.55  24734.40  24728.45  24733.55


In [4]:
import pandas as pd
import numpy as np

# ============================================================
# 1) Build DAILY OHLC from your 1-minute bars (09:15–15:25)
# ============================================================
def make_daily_from_1min(spot_1min: pd.DataFrame) -> pd.DataFrame:
    """
    spot_1min: columns ['Datetime','Open','High','Low','Close'] (IST 09:15–15:25)
    Returns daily OHLC with the Open taken from the 09:15 bar (or first bar >= 09:15).
    """
    df = spot_1min.copy()
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Date'] = df['Datetime'].dt.date
    df['Time'] = df['Datetime'].dt.time

    # Daily OHLC using per-day slices to ensure Open is 09:15 bar
    rows = []
    for d, day in df.groupby('Date'):
        day = day.sort_values('Datetime')
        # open = first bar at/after 09:15
        day_open_row = day.iloc[0]
        O = float(day_open_row['Open'])
        H = float(day['High'].max())
        L = float(day['Low'].min())
        C = float(day.iloc[-1]['Close'])
        rows.append({'Date': pd.to_datetime(d), 'Open': O, 'High': H, 'Low': L, 'Close': C})
    daily = pd.DataFrame(rows).sort_values('Date').reset_index(drop=True)
    return daily

# ============================================================
# 2) Compute NR7 flags and Stretch (SMA10 of Noise), shifted
# ============================================================
def compute_nr7_and_stretch(daily_df: pd.DataFrame, noise_win: int = 10) -> pd.DataFrame:
    """
    daily_df: ['Date','Open','High','Low','Close']
    Returns daily features including:
      - Range, NR7 flag, NR7_prev (acts as trade flag for the NEXT day)
      - Noise and Stretch_today (SMA10(Noise) shifted so it's known at today's open)
    """
    d = daily_df.copy()
    d['Date'] = pd.to_datetime(d['Date']).dt.normalize()
    d['Range'] = d['High'] - d['Low']

    # NR7: today's range is the SMALLEST of the last 7 ranges (incl. today)
    d['NR7'] = d['Range'].eq(d['Range'].rolling(7, min_periods=7).min())

    # Trade on the day AFTER NR7:
    d['NR7_prev'] = d['NR7'].shift(1).fillna(False)

    # Crabel Noise & Stretch (use past 10 days; known at today's open)
    d['Noise'] = pd.concat([(d['High']-d['Open']), (d['Open']-d['Low'])], axis=1).min(axis=1)
    d['Stretch_today'] = d['Noise'].rolling(noise_win, min_periods=noise_win).mean().shift(1)
    return d[['Date','Open','High','Low','Close','Range','NR7','NR7_prev','Noise','Stretch_today']]

# ============================================================
# 3) Backtest: NR7-gated Open ± (1.15*Stretch) with 150 target or EOD
# ============================================================
def _clock(hhmm: str):
    return pd.to_datetime(hhmm, format="%H:%M").time()

def backtest_nr7_orb_open_stretch(
    spot_1min: pd.DataFrame,
    daily_feats: pd.DataFrame,
    k: float = 1.15,
    market_open: str = "09:15",
    market_close: str = "15:25",
    target_points: float = 150.0,
    skip_weekdays: tuple = ("Monday",),
    be_after_minutes: int = 60,          # NEW: move SL to breakeven after this many minutes
):
    """
    NR7-gated Crabel ORB with OCO at Open ± k*Stretch.
    - Skip entries on weekdays in `skip_weekdays`.
    - Exit at target or EOD, with SL = opposite trigger UNTIL `be_after_minutes` passes,
      then SL is moved to breakeven (entry price).
    """
    df = spot_1min.copy()
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Date'] = df['Datetime'].dt.date
    df['Time'] = df['Datetime'].dt.time

    feats = daily_feats.copy()
    feats['weekday'] = feats['Date'].dt.day_name()

    oclk = pd.to_datetime(market_open, format="%H:%M").time()
    cclk = pd.to_datetime(market_close, format="%H:%M").time()

    results = []

    # ---- Exit engine with dynamic breakeven stop after be_after_minutes ----
    def evaluate_path(fwd, side, entry_px, stop_level, target_px, entry_time):
        """
        fwd: 1-min bars from (and including) entry bar time onward
        After `be_after_minutes`, SL moves to entry price (breakeven).
        Conservative tie-break: if stop and target touch in same bar -> STOP.
        """
        entry_time = pd.to_datetime(entry_time)
        be_ts = entry_time + pd.Timedelta(minutes=be_after_minutes)

        # Favourable-threshold flags
        if side == 'LONG':
            highs = fwd['High'].cummax()
            reached = lambda pts: bool((highs >= entry_px + pts).any())
        else:
            lows = fwd['Low'].cummin()
            reached = lambda pts: bool((lows <= entry_px - pts).any())

        flags = {
            'hit_50':  reached(50),
            'hit_75':  reached(75),
            'hit_100': reached(100),
            'hit_150': reached(150),
            'hit_200': reached(200),
        }

        for _, bar in fwd.iterrows():
            hi, lo, ts = float(bar['High']), float(bar['Low']), pd.to_datetime(bar['Datetime'])

            # Effective stop: original until be_ts, then breakeven (entry_px)
            eff_stop = stop_level if ts < be_ts else entry_px

            if side == 'LONG':
                stop_touched   = (lo <= eff_stop)
                target_touched = (hi >= target_px)
                if stop_touched and target_touched:
                    return ts, float(eff_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(eff_stop), 'STOP', flags
            else:  # SHORT
                stop_touched   = (hi >= eff_stop)
                target_touched = (lo <= target_px)
                if stop_touched and target_touched:
                    return ts, float(eff_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(eff_stop), 'STOP', flags

        # No exit hit → EOD at last bar before/at close
        last_bar = fwd[fwd['Datetime'].dt.time <= cclk]
        last_bar = last_bar.iloc[-1] if not last_bar.empty else fwd.iloc[-1]
        return pd.to_datetime(last_bar['Datetime']), float(last_bar['Close']), 'EOD', flags

    # ---- Iterate trade days ----
    for _, row in feats.iterrows():
        if row.get('weekday') in skip_weekdays:
            continue
        if not bool(row['NR7_prev']):
            continue
        if pd.isna(row['Stretch_today']) or row['Stretch_today'] <= 0:
            continue

        d = row['Date'].date()
        day = df[df['Date'] == d].sort_values('Datetime')
        if day.empty:
            continue

        after_open = day[day['Time'] >= oclk].copy()
        if after_open.empty:
            continue

        open_row = after_open.iloc[0]
        O = float(open_row['Open'])
        stretch = float(row['Stretch_today']) * float(k)

        buy_trig  = O + stretch
        sell_trig = O - stretch

        path = day[day['Datetime'] > open_row['Datetime']].copy()
        if path.empty:
            continue

        long_hit  = path[path['High'] >= buy_trig].head(1)
        short_hit = path[path['Low']  <= sell_trig].head(1)

        if long_hit.empty and short_hit.empty:
            continue

        if not long_hit.empty and not short_hit.empty:
            side = 'LONG' if long_hit.iloc[0]['Datetime'] <= short_hit.iloc[0]['Datetime'] else 'SHORT'
        elif not long_hit.empty:
            side = 'LONG'
        else:
            side = 'SHORT'

        entry_time = (long_hit.iloc[0]['Datetime'] if side=='LONG' else short_hit.iloc[0]['Datetime'])
        entry_px   = (buy_trig if side=='LONG' else sell_trig)
        stop_level = (sell_trig if side=='LONG' else buy_trig)   # original opposite-trigger stop
        target_px  = (entry_px + target_points) if side=='LONG' else (entry_px - target_points)

        fwd = path[path['Datetime'] >= entry_time].copy()
        if fwd.empty:
            continue

        exit_time, exit_px, exit_reason, flags = evaluate_path(
            fwd=fwd, side=side, entry_px=entry_px, stop_level=stop_level,
            target_px=target_px, entry_time=entry_time
        )

        pnl = (exit_px - entry_px) if side=='LONG' else (entry_px - exit_px)
        win = pnl > 0

        results.append({
            'Date': row['Date'].normalize(), 'weekday': row['weekday'],
            'NR7_prev': bool(row['NR7_prev']),
            'Open_0915': O,
            'Stretch': float(row['Stretch_today']),
            'k_used': k,
            'buy_stop': buy_trig, 'sell_stop': sell_trig,
            'side': side, 'entry_time': pd.to_datetime(entry_time), 'entry_px': entry_px,
            'orig_stop_level': stop_level,
            'be_armed_after_min': be_after_minutes,    # informational
            'target_px': target_px,
            'exit_time': exit_time, 'exit_px': exit_px, 'exit_reason': exit_reason,
            'pnl_points': pnl, 'win': win,
            'hit_50': flags['hit_50'], 'hit_75': flags['hit_75'],
            'hit_100': flags['hit_100'], 'hit_150': flags['hit_150'],
            'hit_200': flags['hit_200'],
        })

    trades = pd.DataFrame(results).sort_values(['Date','entry_time']).reset_index(drop=True)

    if trades.empty:
        summary = pd.Series({'trades':0, 'wins':0, 'win_rate':np.nan,
                             'total_points':0.0, 'avg_points':np.nan, 'median_points':np.nan})
    else:
        wins = int(trades['win'].sum())
        summary = pd.Series({
            'trades': len(trades),
            'wins': wins,
            'win_rate': wins / len(trades),
            'total_points': trades['pnl_points'].sum(),
            'avg_points': trades['pnl_points'].mean(),
            'median_points': trades['pnl_points'].median()
        })
    return trades, summary




In [8]:
# You already built spot_1min (09:15–15:25) as shown in your snippet
daily = make_daily_from_1min(spot_1min)

feats = compute_nr7_and_stretch(daily, noise_win=10)   # NR7 flags + Stretch (shifted)

trades, summary = backtest_nr7_orb_open_stretch(
    spot_1min, feats,
    k=1.5, market_open="09:15", market_close="15:25",
    target_points=200
)

print(summary)        # trades, wins, win_rate, total/avg/median points
print(trades.head())  # includes Stretch, weekday, hit_50/75/100/150/200, etc.


/tmp/ipykernel_316997/3583682481.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d['NR7_prev'] = d['NR7'].shift(1).fillna(False)


trades            127.000000
wins               23.000000
win_rate            0.181102
total_points     2741.635000
avg_points         21.587677
median_points       0.000000
dtype: float64
        Date    weekday  NR7_prev  Open_0915  Stretch  k_used    buy_stop  \
0 2021-06-16  Wednesday      True   15848.90   32.005     1.5  15896.9075   
1 2021-06-25     Friday      True   15844.45   30.250     1.5  15889.8250   
2 2021-07-02     Friday      True   15711.20   24.005     1.5  15747.2075   
3 2021-07-06    Tuesday      True   15821.85   27.410     1.5  15862.9650   
4 2021-07-14  Wednesday      True   15811.65   28.655     1.5  15854.6325   

    sell_stop   side          entry_time  ...           exit_time     exit_px  \
0  15800.8925  SHORT 2021-06-16 10:21:00  ... 2021-06-16 11:37:00  15800.8925   
1  15799.0750  SHORT 2021-06-25 09:21:00  ... 2021-06-25 10:31:00  15799.0750   
2  15675.1925  SHORT 2021-07-02 09:20:00  ... 2021-07-02 10:20:00  15675.1925   
3  15780.7350   LONG 202

In [9]:
trades.to_excel("trades.xlsx", index=False)  

In [10]:

#LOAD AND SAVE OPTION DATA SEPARATELY
years_to_load = [2021, 2022, 2023, 2024, 2025]

outdir = "./options_data_yearwise"
os.makedirs(outdir, exist_ok=True)

for year in years_to_load:
    print(f"\n--- Loading option data for year: {year} ---")

    # === NEAREST EXPIRY ===
    options_files = glob.glob(f"/home/newberry3/main/Data/NIFTY/NIFTY_{year}*.pkl")
    if not options_files:
        print(f"No .pkl files found in /main/Data/NIFTY for year {year}")
        continue

    all_cols = set()
    dfs = []
    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        all_cols.update(df.columns)
    all_cols = sorted(list(all_cols))

    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        for col in all_cols:
            if col not in df.columns:
                df[col] = pd.NA
        df = df[all_cols]
        dfs.append(df)
    options_df = pd.concat(dfs, ignore_index=True)
    for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
        if col in options_df.columns:
            options_df[col] = options_df[col].astype(str)
    options_df['DateTime'] = pd.to_datetime(options_df['Date'] + " " + options_df['Time'], errors='coerce')
    options_df = options_df.dropna(subset=['DateTime'])
    cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date','Time']
    options_df = options_df.drop(columns=[col for col in cols_to_drop if col in options_df.columns])
    cols = ['DateTime'] + [col for col in options_df.columns if col != 'DateTime']
    options_df = options_df[cols]
    options_df['StrikePrice'] = options_df['StrikePrice'].astype(float)
    options_df['Type'] = options_df['Type'].str.strip().str.upper()
    options_df['ExpiryDate'] = pd.to_datetime(options_df['ExpiryDate']).dt.normalize()

    # --- Save nearest expiry for this year ---
    nearest_path = os.path.join(outdir, f"options_nearest_{year}.parquet")
    options_df.to_parquet(nearest_path, index=False)
    print(f"✅ Saved NEAREST expiry options for {year} to {nearest_path}")

print(f"\nAll options data saved to {outdir}")



--- Loading option data for year: 2021 ---


/tmp/ipykernel_316997/3370578563.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  options_df = pd.concat(dfs, ignore_index=True)


✅ Saved NEAREST expiry options for 2021 to ./options_data_yearwise/options_nearest_2021.parquet

--- Loading option data for year: 2022 ---
✅ Saved NEAREST expiry options for 2022 to ./options_data_yearwise/options_nearest_2022.parquet

--- Loading option data for year: 2023 ---
✅ Saved NEAREST expiry options for 2023 to ./options_data_yearwise/options_nearest_2023.parquet

--- Loading option data for year: 2024 ---
✅ Saved NEAREST expiry options for 2024 to ./options_data_yearwise/options_nearest_2024.parquet

--- Loading option data for year: 2025 ---
✅ Saved NEAREST expiry options for 2025 to ./options_data_yearwise/options_nearest_2025.parquet

All options data saved to ./options_data_yearwise


In [11]:
#REQUIRED FOR ALL
# BS/IV functions

# --- Black-Scholes Option Price ---
def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None
    
# --- Implied Volatility from Option Price ---
def implied_volatility(option_price, S, K, T, r, option_type):
    """
    Uses Brent's method to find implied volatility from the market price.
    """
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None
    
# --- Black-Scholes Greeks ---
def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None

    # Use synthetic future price
    F = S * np.exp(r * T)

    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
    vega = F * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Delta': round(delta, 5),
        'Gamma': round(gamma, 5),
        'Vega': round(vega, 5),
        'Theta': round(theta, 5),
        'Rho': round(rho, 5)
    }

# --- Time to Expiry (Fractional) ---
def calculate_time_to_expiry(manual_datetime_str, expiry_date_str):
    now = datetime.strptime(manual_datetime_str, "%Y-%m-%d %H:%M:%S")
    expiry_date = datetime.strptime(expiry_date_str, "%d-%m-%y").date()

    market_open = time(9, 15)
    market_close = time(15, 30)
    today = now.date()
    days_left = (expiry_date - today).days

    if days_left <= 0:
        days_left += 1

    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)

    if current_minutes_since_open < 0:
        T = round(days_left / 365, 6)
    elif current_minutes_since_open >= total_trading_minutes:
        T = round(max(0, (days_left - 1) / 365), 6)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = round((days_left - fraction_of_day_passed) / 365, 6)

    print(f"\nManual Time Entered: {now}")
    print(f"Expiry Date: {expiry_date}")
    print(f"Days to Expiry (with fraction): {days_left - (fraction_of_day_passed if 0 <= current_minutes_since_open < total_trading_minutes else 0):.6f}")
    print(f"Time to Expiry in Years (T): {T}")

    return T

In [15]:
import pandas as pd
import numpy as np

# =========================
# Helpers
# =========================
def _calc_T_years(entry_ts: pd.Timestamp, expiry_dt: pd.Timestamp) -> float:
    """Time to expiry in years (ACT/365)."""
    secs = max(0.0, (pd.to_datetime(expiry_dt) - pd.to_datetime(entry_ts)).total_seconds())
    return secs / (365.0 * 24 * 3600)

def _normalize_side(side_val) -> str:
    """Return 'BULLISH' or 'BEARISH' from LONG/SHORT/etc."""
    s = str(side_val).strip().upper()
    if s in ("LONG", "BULLISH", "BUY", "L"):
        return "BULLISH"
    elif s in ("SHORT", "BEARISH", "SELL", "S"):
        return "BEARISH"
    raise ValueError(f"Unrecognized side: {side_val}")

def _round_to_step(x: float, step: int = 50) -> float:
    """Round x to nearest strike step (default 50)."""
    return round(x / step) * step

# =========================
# Core: pick strike for ONE trade
# =========================
def pick_nearest_expiry_atm_band(
    trade_row: pd.Series,
    options_df: pd.DataFrame,
    *,
    step: int = 50,
    band_points: int = 100,      # consider ATM±50, ±100
    price_col: str = "Open",
    target_abs_delta: float = 0.70,
    r: float = 0.066,
    time_window_min: int = 1,
    verbose: bool = False,
):
    """
    For ONE ORB trade row:
      - Use NEAREST expiry >= entry time.
      - Determine ATM from spot (rounded to 'step'); consider strikes: ATM±50, ATM±100.
      - If side is LONG/BULLISH -> CE (target +0.70). If SHORT/BEARISH -> PE (target -0.70).
      - Choose strike whose delta is closest to target at entry minute.
    Requires user functions: implied_volatility(...), black_scholes_greeks(...)
    """
    # 1) Extract trade fields from your ORB code columns
    entry_ts = pd.to_datetime(trade_row.get('entry_time') or trade_row.get('Entry Time'))
    spot     = float(trade_row.get('entry_px')   or trade_row.get('Entry Price'))
    bias     = _normalize_side(trade_row.get('side') or trade_row.get('Side'))
    opt_type = 'CE' if bias == 'BULLISH' else 'PE'
    target_delta = +target_abs_delta if opt_type == 'CE' else -target_abs_delta

    # 2) Sanity & types
    if options_df is None or options_df.empty:
        if verbose: print(f"[WARN] No options chain provided for entry {entry_ts}.")
        return None

    df = options_df.copy()
    df['StrikePrice'] = pd.to_numeric(df['StrikePrice'], errors='coerce')
    df['ExpiryDate']  = pd.to_datetime(df['ExpiryDate'])
    df['DateTime']    = pd.to_datetime(df['DateTime'])
    df['Type']        = df['Type'].astype(str).str.upper()

    # 3) NEAREST expiry ON/AFTER entry
    expiry_list = (df.loc[df['DateTime'] >= entry_ts, 'ExpiryDate']
                     .drop_duplicates().sort_values())
    if expiry_list.empty:
        if verbose: print(f"[DEBUG] No expiry >= {entry_ts}.")
        return None
    expiry = pd.to_datetime(expiry_list.iloc[0])

    # 4) Snapshot at entry minute ± window for chosen expiry & option type
    t_win = pd.Timedelta(minutes=time_window_min)
    snap = df[
        (df['Type'] == opt_type) &
        (df['ExpiryDate'] == expiry) &
        (df['DateTime'].between(entry_ts - t_win, entry_ts + t_win)) &
        (df['DateTime'].dt.floor('min') == entry_ts.floor('min'))
    ].copy()
    if snap.empty:
        if verbose:
            print(f"[DEBUG] No rows at entry minute for {opt_type}, "
                  f"expiry={expiry.date()} @ {entry_ts}.")
        return None

    # 5) Build candidate strikes: ATM±50, ±100 (if present in snapshot)
    atm = _round_to_step(spot, step=step)
    candidates = [atm - band_points, atm - step, atm, atm + step, atm + band_points]
    avail = sorted(set(snap['StrikePrice'].dropna().astype(float).tolist()))
    cand_avail = [k for k in candidates if k in avail]
    if not cand_avail:
        if verbose:
            print(f"[DEBUG] No candidate strikes available at entry: "
                  f"ATM basis={atm}, candidates={candidates}, present={avail[:10]}...")
        return None

    # 6) Choose strike with |delta - target| minimal among candidates
    best, best_diff = None, np.inf

    # Time to expiry helper (use user's calculate_time_to_expiry if available)
    try:
        _ = calculate_time_to_expiry  # noqa
        def _T(entry_ts_local, expiry_dt_local):
            return calculate_time_to_expiry(pd.to_datetime(entry_ts_local).strftime("%Y-%m-%d %H:%M:%S"),
                                            pd.to_datetime(expiry_dt_local).strftime("%d-%m-%y"))
    except NameError:
        def _T(entry_ts_local, expiry_dt_local):
            return _calc_T_years(entry_ts_local, expiry_dt_local)

    bs_kind = 'call' if opt_type == 'CE' else 'put'

    for K in cand_avail:
        row_k = snap.loc[snap['StrikePrice'] == K]
        if row_k.empty:
            continue
        px = float(row_k.iloc[0][price_col] if price_col in row_k.columns else row_k.iloc[0]['Open'])

        T = _T(entry_ts, expiry)
        if not (T > 0):
            if verbose: print(f"  [DEBUG] Non-positive T for {expiry.date()} at K={K}.")
            continue

        iv = implied_volatility(px, spot, K, T, r, bs_kind)
        if iv is None or iv <= 0:
            if verbose: print(f"  [DEBUG] IV fail at K={K:.0f}, px={px:.2f}.")
            continue

        greeks = black_scholes_greeks(spot, K, T, r, iv, bs_kind)
        if greeks is None or 'Delta' not in greeks:
            if verbose: print(f"  [DEBUG] Greeks fail at K={K:.0f}, IV={iv:.4f}.")
            continue

        delta = float(greeks['Delta'])
        diff = abs(delta - target_delta)
        if verbose:
            print(f"  [DBG] {opt_type} K={K:.0f} px={px:.2f} Δ={delta:.4f} |Δ-Δ*|={diff:.4f} IV={iv:.4f}")
        if diff < best_diff:
            best_diff = diff
            best = {
                'entry_time': entry_ts,
                'side': bias,
                'spot_entry': spot,
                'option_type': opt_type,
                'expiry': expiry,
                'atm_basis': atm,
                'strike': K,
                'option_price': px,
                'iv': iv,
                'delta': delta,
                'target_delta': target_delta
            }

    if best is None and verbose:
        print(f"[DEBUG] No valid IV/Greeks among candidates {cand_avail} for {entry_ts}.")
    return best

# =========================
# Batch: map over trades
# =========================
def build_orb_option_entries_nearest(
    trades: pd.DataFrame,
    options_by_year_or_df,
    *,
    step: int = 50,
    band_points: int = 100,
    price_col: str = "Open",
    target_abs_delta: float = 0.70,
    r: float = 0.066,
    time_window_min: int = 1,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Apply pick_nearest_expiry_atm_band to all trades.
    - options_by_year_or_df can be:
        * a dict: {year: options_df_for_that_year}
        * a single DataFrame with multiple years
    """
    def _pick_col(df, *candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise ValueError(f"None of the expected columns {candidates} found in trades df: {list(df.columns)}")

    t = trades.copy()

    # --- Normalize required columns (choose whichever exists) ---
    entry_time_col = _pick_col(t, 'entry_time', 'Entry Time')
    entry_px_col   = _pick_col(t, 'entry_px', 'Entry Price')
    side_col       = _pick_col(t, 'side', 'Side')

    # Standardize column names used downstream
    if entry_time_col != 'entry_time':
        t = t.rename(columns={entry_time_col: 'entry_time'})
    if entry_px_col != 'entry_px':
        t = t.rename(columns={entry_px_col: 'entry_px'})
    if side_col != 'side':
        t = t.rename(columns={side_col: 'side'})

    # Ensure types
    t['entry_time'] = pd.to_datetime(t['entry_time'])
    t['entry_px']   = pd.to_numeric(t['entry_px'], errors='coerce')
    t['side']       = t['side'].astype(str)

    t['entry_year'] = t['entry_time'].dt.year

    results = []

    def _get_chain_for_year(y):
        if isinstance(options_by_year_or_df, dict):
            return options_by_year_or_df.get(y)
        else:
            # single DF provided; we'll pass it through (pick function filters by time/expiry)
            return options_by_year_or_df

    for year, chunk in t.groupby('entry_year'):
        chain = _get_chain_for_year(year)
        if chain is None or (isinstance(chain, pd.DataFrame) and chain.empty):
            if verbose: print(f"[WARN] Missing/empty options chain for {year}; skipping.")
            continue

        for _, tr in chunk.iterrows():
            sel = pick_nearest_expiry_atm_band(
                tr, chain,
                step=step, band_points=band_points, price_col=price_col,
                target_abs_delta=target_abs_delta, r=r, time_window_min=time_window_min,
                verbose=verbose
            )
            if sel is not None:
                results.append(sel)

    return pd.DataFrame(results)


In [16]:
import os, pandas as pd

outdir = "./options_data_yearwise"  # same as in your save step
years_to_load = [2021, 2022, 2023, 2024, 2025]

# Build options_by_year from the saved files
options_by_year = {}
for y in years_to_load:
    fp = os.path.join(outdir, f"options_nearest_{y}.parquet")
    if os.path.exists(fp):
        options_by_year[y] = pd.read_parquet(fp)

# Use your dict directly
chosen = build_orb_option_entries_nearest(
    trades=trades,
    options_by_year_or_df=options_by_year,
    step=50,
    band_points=100,
    price_col="Open",
    target_abs_delta=0.70,
    r=0.066,
    time_window_min=1,
    verbose=False
)

trades_with_opts = trades.merge(chosen, on=['entry_time','side'], how='left')
trades_with_opts.to_excel('trades_with_option_selection.xlsx', index=False)
print("Saved trades_with_option_selection.xlsx")



Manual Time Entered: 2021-06-16 10:21:00
Expiry Date: 2021-06-17
Days to Expiry (with fraction): 0.824000
Time to Expiry in Years (T): 0.002258

Manual Time Entered: 2021-06-16 10:21:00
Expiry Date: 2021-06-17
Days to Expiry (with fraction): 0.824000
Time to Expiry in Years (T): 0.002258

Manual Time Entered: 2021-06-16 10:21:00
Expiry Date: 2021-06-17
Days to Expiry (with fraction): 0.824000
Time to Expiry in Years (T): 0.002258

Manual Time Entered: 2021-06-16 10:21:00
Expiry Date: 2021-06-17
Days to Expiry (with fraction): 0.824000
Time to Expiry in Years (T): 0.002258

Manual Time Entered: 2021-06-16 10:21:00
Expiry Date: 2021-06-17
Days to Expiry (with fraction): 0.824000
Time to Expiry in Years (T): 0.002258

Manual Time Entered: 2021-06-25 09:21:00
Expiry Date: 2021-07-01
Days to Expiry (with fraction): 5.984000
Time to Expiry in Years (T): 0.016395

Manual Time Entered: 2021-06-25 09:21:00
Expiry Date: 2021-07-01
Days to Expiry (with fraction): 5.984000
Time to Expiry in Years

In [57]:
# CHANGES MADE (as requested):
# 1) ENTRY RULE: Enter ONLY when the 1-min bar's CLOSE crosses the bound:
#       - LONG when Close >= (Open + k*Stretch)
#       - SHORT when Close <= (Open − k*Stretch)
#    (Previously we triggered on High/Low touches.)
#
# 2) STOP MANAGEMENT (two possible upgrades, choose the one that happens FIRST and lock it):
#    a) TIME-BASED: exactly 60 minutes (be_after_minutes) after entry → move SL to BREAKEVEN (entry price).
#    b) PRICE-BASED: once price moves +50 pts in favor → move SL to lock +50 pts profit
#       (i.e., for LONG: SL = entry + 50; for SHORT: SL = entry − 50).
#    If both conditions occur in the same bar, we conservatively choose BREAKEVEN.
#
# 3) Everything else unchanged: opposite-trigger is the initial SL; optional target_points exit; EOD exit; 
#    one trade/day; Monday skip; threshold flags (hit_50/75/100/150/200); conservative tie-break
#    (if STOP and TARGET touch in the same bar, count it as STOP).

def backtest_nr7_orb_open_stretch(
    spot_1min: pd.DataFrame,
    daily_feats: pd.DataFrame,
    k: float = 1.15,
    market_open: str = "09:15",
    market_close: str = "15:25",
    target_points: float = 150.0,
    skip_weekdays: tuple = ("Monday",),
    be_after_minutes: int = 60,          # move SL to breakeven after this many minutes
):
    df = spot_1min.copy()
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Date'] = df['Datetime'].dt.date
    df['Time'] = df['Datetime'].dt.time

    feats = daily_feats.copy()
    feats['weekday'] = feats['Date'].dt.day_name()

    oclk = pd.to_datetime(market_open, format="%H:%M").time()
    cclk = pd.to_datetime(market_close, format="%H:%M").time()

    results = []

    def evaluate_path_with_locked_stop(fwd, side, entry_px, orig_stop, target_px, entry_time):
        """
        After entry, there are two potential stop upgrades:
          - time-based BE at entry_time + be_after_minutes
          - price-based +50 lock once move in favor reaches +50
        Whichever happens FIRST becomes the locked stop for the rest of the day.
        Conservative tie-breaks:
          * If STOP and TARGET touch in the same bar -> STOP.
          * If BE-time and +50 occur in the same bar -> choose BREAKEVEN.
        """
        entry_time = pd.to_datetime(entry_time)
        be_ts = entry_time + pd.Timedelta(minutes=be_after_minutes)

        # For reporting only (favourable thresholds)
        if side == 'LONG':
            highs_cum = fwd['High'].cummax()
            flags = {
                'hit_50':  bool((highs_cum >= entry_px + 50).any()),
                'hit_75':  bool((highs_cum >= entry_px + 75).any()),
                'hit_100': bool((highs_cum >= entry_px + 100).any()),
                'hit_150': bool((highs_cum >= entry_px + 150).any()),
                'hit_200': bool((highs_cum >= entry_px + 200).any()),
            }
        else:
            lows_cum = fwd['Low'].cummin()
            flags = {
                'hit_50':  bool((lows_cum <= entry_px - 50).any()),
                'hit_75':  bool((lows_cum <= entry_px - 75).any()),
                'hit_100': bool((lows_cum <= entry_px - 100).any()),
                'hit_150': bool((lows_cum <= entry_px - 150).any()),
                'hit_200': bool((lows_cum <= entry_px - 200).any()),
            }

        stop_locked = False
        locked_stop = None

        for _, bar in fwd.iterrows():
            hi, lo, ts = float(bar['High']), float(bar['Low']), pd.to_datetime(bar['Datetime'])

            # Check upgrade events (not yet locked)
            if not stop_locked:
                hit_50_now = (hi >= entry_px + 50) if side == 'LONG' else (lo <= entry_px - 50)
                be_now = (ts >= be_ts)
                if hit_50_now and be_now:
                    # same bar: choose BE (conservative)
                    stop_locked = True
                    locked_stop = entry_px
                elif hit_50_now:
                    stop_locked = True
                    locked_stop = entry_px + 50 if side == 'LONG' else entry_px - 50
                elif be_now:
                    stop_locked = True
                    locked_stop = entry_px

            # Effective stop for this bar
            eff_stop = locked_stop if stop_locked else orig_stop

            # Exit checks with conservative tie-break (STOP over TARGET)
            if side == 'LONG':
                stop_touched   = (lo <= eff_stop)
                target_touched = (hi >= target_px)
                if stop_touched and target_touched:
                    return ts, float(eff_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(eff_stop), 'STOP', flags
            else:
                stop_touched   = (hi >= eff_stop)
                target_touched = (lo <= target_px)
                if stop_touched and target_touched:
                    return ts, float(eff_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(eff_stop), 'STOP', flags

        # EOD exit if nothing hit
        last_bar = fwd[fwd['Datetime'].dt.time <= cclk]
        last_bar = last_bar.iloc[-1] if not last_bar.empty else fwd.iloc[-1]
        return pd.to_datetime(last_bar['Datetime']), float(last_bar['Close']), 'EOD', flags

    # ---- Iterate trade days ----
    for _, row in feats.iterrows():
        if row.get('weekday') in skip_weekdays:
            continue
        if not bool(row['NR7_prev']):
            continue
        if pd.isna(row['Stretch_today']) or row['Stretch_today'] <= 0:
            continue

        d = row['Date'].date()
        day = df[df['Date'] == d].sort_values('Datetime')
        if day.empty:
            continue

        after_open = day[day['Time'] >= oclk].copy()
        if after_open.empty:
            continue

        open_row = after_open.iloc[0]
        O = float(open_row['Open'])
        stretch = float(row['Stretch_today']) * float(k)

        buy_trig  = O + stretch
        sell_trig = O - stretch

        # ENTRY ONLY WHEN CLOSE CROSSES THE BOUNDS (first bar after 09:15)
        path = day[day['Datetime'] > open_row['Datetime']].copy()
        if path.empty:
            continue

        long_hit  = path[path['Close'] >= buy_trig].head(1)
        short_hit = path[path['Close'] <= sell_trig].head(1)

        if long_hit.empty and short_hit.empty:
            continue

        if not long_hit.empty and not short_hit.empty:
            side = 'LONG' if long_hit.iloc[0]['Datetime'] <= short_hit.iloc[0]['Datetime'] else 'SHORT'
        elif not long_hit.empty:
            side = 'LONG'
        else:
            side = 'SHORT'

        # Conservative fill at trigger level on the CLOSE-cross bar
        entry_row = long_hit.iloc[0] if side == 'LONG' else short_hit.iloc[0]
        entry_time = entry_row['Datetime']
        entry_px   = buy_trig if side == 'LONG' else sell_trig

        # Initial stop is opposite trigger; target remains as parameterized
        orig_stop = sell_trig if side == 'LONG' else buy_trig
        target_px = (entry_px + target_points) if side == 'LONG' else (entry_px - target_points)

        fwd = path[path['Datetime'] >= entry_time].copy()
        if fwd.empty:
            continue

        exit_time, exit_px, exit_reason, flags = evaluate_path_with_locked_stop(
            fwd=fwd, side=side, entry_px=entry_px, orig_stop=orig_stop,
            target_px=target_px, entry_time=entry_time
        )

        pnl = (exit_px - entry_px) if side == 'LONG' else (entry_px - exit_px)
        win = pnl > 0

        results.append({
            'Date': row['Date'].normalize(), 'weekday': row['weekday'],
            'NR7_prev': bool(row['NR7_prev']),
            'Open_0915': O,
            'Stretch': float(row['Stretch_today']),
            'k_used': k,
            'buy_stop': buy_trig, 'sell_stop': sell_trig,
            'side': side, 'entry_time': pd.to_datetime(entry_time), 'entry_px': entry_px,
            'orig_stop_level': orig_stop,
            'be_after_minutes': be_after_minutes,
            'locked_stop_rule': 'first_of(BE_time, +50_pts)',
            'target_px': target_px,
            'exit_time': exit_time, 'exit_px': exit_px, 'exit_reason': exit_reason,
            'pnl_points': pnl, 'win': win,
            'hit_50': flags['hit_50'], 'hit_75': flags['hit_75'],
            'hit_100': flags['hit_100'], 'hit_150': flags['hit_150'],
            'hit_200': flags['hit_200'],
        })

    trades = pd.DataFrame(results).sort_values(['Date','entry_time']).reset_index(drop=True)

    if trades.empty:
        summary = pd.Series({'trades':0, 'wins':0, 'win_rate':np.nan,
                             'total_points':0.0, 'avg_points':np.nan, 'median_points':np.nan})
    else:
        wins = int(trades['win'].sum())
        summary = pd.Series({
            'trades': len(trades),
            'wins': wins,
            'win_rate': wins / len(trades),
            'total_points': trades['pnl_points'].sum(),
            'avg_points': trades['pnl_points'].mean(),
            'median_points': trades['pnl_points'].median()
        })
    return trades, summary
# You already built spot_1min (09:15–15:25) as shown in your snippet
daily = make_daily_from_1min(spot_1min)

feats = compute_nr7_and_stretch(daily, noise_win=10)   # NR7 flags + Stretch (shifted)

trades, summary = backtest_nr7_orb_open_stretch(
    spot_1min, feats,
    k=1.25, market_open="09:15", market_close="15:25",
    target_points=200
)

print(summary)        # trades, wins, win_rate, total/avg/median points
print(trades.head())  # includes Stretch, weekday, hit_50/75/100/150/200, etc.


/tmp/ipykernel_315450/3583682481.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d['NR7_prev'] = d['NR7'].shift(1).fillna(False)


trades            131.000000
wins               45.000000
win_rate            0.343511
total_points     2206.406250
avg_points         16.842796
median_points       0.000000
dtype: float64
        Date    weekday  NR7_prev  Open_0915  Stretch  k_used     buy_stop  \
0 2021-06-16  Wednesday      True   15848.90   32.005    1.25  15888.90625   
1 2021-06-25     Friday      True   15844.45   30.250    1.25  15882.26250   
2 2021-07-02     Friday      True   15711.20   24.005    1.25  15741.20625   
3 2021-07-06    Tuesday      True   15821.85   27.410    1.25  15856.11250   
4 2021-07-14  Wednesday      True   15811.65   28.655    1.25  15847.46875   

     sell_stop   side          entry_time  ...           exit_time  \
0  15808.89375  SHORT 2021-06-16 10:21:00  ... 2021-06-16 13:57:00   
1  15806.63750  SHORT 2021-06-25 09:21:00  ... 2021-06-25 10:32:00   
2  15681.19375  SHORT 2021-07-02 09:20:00  ... 2021-07-02 10:27:00   
3  15787.58750   LONG 2021-07-06 09:18:00  ... 2021-07-06 10:1

In [65]:
trades.to_excel("trades_trail_sl.xlsx", index=False)  

In [64]:
# CHANGES MADE NOW:
# - ENTRY: only when the 1-min bar CLOSE crosses the trigger (same as your last version).
# - STOP UPGRADES (monotonic, intraday):
#     * Earliest of:
#         (i) time-based BE at entry_time + be_after_minutes, OR
#         (ii) price-based +50 in favor  --> set SL to entry±50
#       (if both in same bar → choose BE).
#     * Thereafter, TRAIL STEPWISE as price reaches:
#         +100  → SL = entry±100
#         +150  → SL = entry±150
#       (for SHORTs use minus signs). Stop never loosens; it only tightens.
# - Conservative tie-break: if a bar touches both stop and target → count as STOP.

def backtest_nr7_orb_open_stretch(
    spot_1min: pd.DataFrame,
    daily_feats: pd.DataFrame,
    k: float = 1.15,
    market_open: str = "09:15",
    market_close: str = "15:25",
    target_points: float = 200.0,
    skip_weekdays: tuple = ("Monday",),
    be_after_minutes: int = 60,          # time-based BE upgrade
):
    df = spot_1min.copy()
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Date'] = df['Datetime'].dt.date
    df['Time'] = df['Datetime'].dt.time

    feats = daily_feats.copy()
    feats['weekday'] = feats['Date'].dt.day_name()

    oclk = pd.to_datetime(market_open, format="%H:%M").time()
    cclk = pd.to_datetime(market_close, format="%H:%M").time()

    results = []

    def evaluate_path_with_step_trail(fwd, side, entry_px, orig_stop, target_px, entry_time):
        """
        Step-trailing stop:
          - First upgrade = earlier of BE_time or +50 in favor (same-bar clash -> BE).
          - Then trail to +100, then +150, if/when reached.
        """
        entry_time = pd.to_datetime(entry_time)
        be_ts = entry_time + pd.Timedelta(minutes=be_after_minutes)

        # Threshold flags (reporting only, same semantics as before)
        if side == 'LONG':
            highs_cum = fwd['High'].cummax()
            flags = {
                'hit_50':  bool((highs_cum >= entry_px + 50).any()),
                'hit_75':  bool((highs_cum >= entry_px + 75).any()),
                'hit_100': bool((highs_cum >= entry_px + 100).any()),
                'hit_150': bool((highs_cum >= entry_px + 150).any()),
                'hit_200': bool((highs_cum >= entry_px + 200).any()),
            }
        else:
            lows_cum = fwd['Low'].cummin()
            flags = {
                'hit_50':  bool((lows_cum <= entry_px - 50).any()),
                'hit_75':  bool((lows_cum <= entry_px - 75).any()),
                'hit_100': bool((lows_cum <= entry_px - 100).any()),
                'hit_150': bool((lows_cum <= entry_px - 150).any()),
                'hit_200': bool((lows_cum <= entry_px - 200).any()),
            }

        curr_stop = orig_stop
        first_upgrade_done = False

        for _, bar in fwd.iterrows():
            hi, lo, ts = float(bar['High']), float(bar['Low']), pd.to_datetime(bar['Datetime'])

            # ---- 1) Determine candidate upgrades this bar ----
            # Event A: time-based BE active?
            be_now = (ts >= be_ts)
            # Event B/C/D: price-based thresholds hit this bar?
            if side == 'LONG':
                hit_50  = (hi >= entry_px + 50)
                hit_100 = (hi >= entry_px + 100)
                hit_150 = (hi >= entry_px + 150)
            else:
                hit_50  = (lo <= entry_px - 50)
                hit_100 = (lo <= entry_px - 100)
                hit_150 = (lo <= entry_px - 150)

            # ---- 2) First upgrade logic (whichever first: BE_time vs +50) ----
            if not first_upgrade_done:
                if hit_50 and be_now:
                    # same bar: choose BE (conservative)
                    new_stop = entry_px
                    first_upgrade_done = True
                elif hit_50:
                    new_stop = entry_px + 50 if side == 'LONG' else entry_px - 50
                    first_upgrade_done = True
                elif be_now:
                    new_stop = entry_px
                    first_upgrade_done = True
                else:
                    new_stop = curr_stop
            else:
                new_stop = curr_stop

            # ---- 3) Step trail thereafter to +100 and +150 if reached ----
            if first_upgrade_done:
                if side == 'LONG':
                    if hit_100:
                        new_stop = max(new_stop, entry_px + 100)
                    if hit_150:
                        new_stop = max(new_stop, entry_px + 150)
                else:
                    if hit_100:
                        new_stop = min(new_stop, entry_px - 100)
                    if hit_150:
                        new_stop = min(new_stop, entry_px - 150)

            # Ensure stop only tightens
            if side == 'LONG':
                curr_stop = max(curr_stop, new_stop)
            else:
                curr_stop = min(curr_stop, new_stop)

            # ---- 4) Exit checks (STOP over TARGET if both touched in same bar) ----
            if side == 'LONG':
                stop_touched   = (lo <= curr_stop)
                target_touched = (hi >= target_px)
                if stop_touched and target_touched:
                    return ts, float(curr_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(curr_stop), 'STOP', flags
            else:
                stop_touched   = (hi >= curr_stop)
                target_touched = (lo <= target_px)
                if stop_touched and target_touched:
                    return ts, float(curr_stop), 'STOP', flags
                if target_touched:
                    return ts, float(target_px), 'TARGET', flags
                if stop_touched:
                    return ts, float(curr_stop), 'STOP', flags

        # ---- 5) EOD exit if nothing hit ----
        last_bar = fwd[fwd['Datetime'].dt.time <= cclk]
        last_bar = last_bar.iloc[-1] if not last_bar.empty else fwd.iloc[-1]
        return pd.to_datetime(last_bar['Datetime']), float(last_bar['Close']), 'EOD', flags

    # ---- Iterate trade days (same as before) ----
    for _, row in feats.iterrows():
        if row.get('weekday') in skip_weekdays:
            continue
        if not bool(row['NR7_prev']):
            continue
        if pd.isna(row['Stretch_today']) or row['Stretch_today'] <= 0:
            continue

        d = row['Date'].date()
        day = df[df['Date'] == d].sort_values('Datetime')
        if day.empty:
            continue

        after_open = day[day['Time'] >= oclk].copy()
        if after_open.empty:
            continue

        open_row = after_open.iloc[0]
        O = float(open_row['Open'])
        stretch = float(row['Stretch_today']) * float(k)

        buy_trig  = O + stretch
        sell_trig = O - stretch

        # ENTRY ONLY WHEN CLOSE CROSSES (post 09:15 bar onward)
        path = day[day['Datetime'] > open_row['Datetime']].copy()
        if path.empty:
            continue

        long_hit  = path[path['Close'] >= buy_trig].head(1)
        short_hit = path[path['Close'] <= sell_trig].head(1)

        if long_hit.empty and short_hit.empty:
            continue

        if not long_hit.empty and not short_hit.empty:
            side = 'LONG' if long_hit.iloc[0]['Datetime'] <= short_hit.iloc[0]['Datetime'] else 'SHORT'
        elif not long_hit.empty:
            side = 'LONG'
        else:
            side = 'SHORT'

        # Fill at trigger on the close-cross bar (conservative)
        entry_row = long_hit.iloc[0] if side == 'LONG' else short_hit.iloc[0]
        entry_time = entry_row['Datetime']
        entry_px   = buy_trig if side == 'LONG' else sell_trig

        orig_stop = sell_trig if side == 'LONG' else buy_trig
        target_px = (entry_px + target_points) if side == 'LONG' else (entry_px - target_points)

        fwd = path[path['Datetime'] >= entry_time].copy()
        if fwd.empty:
            continue

        exit_time, exit_px, exit_reason, flags = evaluate_path_with_step_trail(
            fwd=fwd, side=side, entry_px=entry_px, orig_stop=orig_stop,
            target_px=target_px, entry_time=entry_time
        )

        pnl = (exit_px - entry_px) if side == 'LONG' else (entry_px - exit_px)
        win = pnl > 0

        results.append({
            'Date': row['Date'].normalize(), 'weekday': row['weekday'],
            'NR7_prev': bool(row['NR7_prev']),
            'Open_0915': O,
            'Stretch': float(row['Stretch_today']),
            'k_used': k,
            'buy_stop': buy_trig, 'sell_stop': sell_trig,
            'side': side, 'entry_time': pd.to_datetime(entry_time), 'entry_px': entry_px,
            'orig_stop_level': orig_stop,
            'be_after_minutes': be_after_minutes,
            'trail_steps': [50,100,150],
            'target_px': target_px,
            'exit_time': exit_time, 'exit_px': exit_px, 'exit_reason': exit_reason,
            'pnl_points': pnl, 'win': win,
            'hit_50': flags['hit_50'], 'hit_75': flags['hit_75'],
            'hit_100': flags['hit_100'], 'hit_150': flags['hit_150'],
            'hit_200': flags['hit_200'],
        })

    trades = pd.DataFrame(results).sort_values(['Date','entry_time']).reset_index(drop=True)

    if trades.empty:
        summary = pd.Series({'trades':0, 'wins':0, 'win_rate':np.nan,
                             'total_points':0.0, 'avg_points':np.nan, 'median_points':np.nan})
    else:
        wins = int(trades['win'].sum())
        summary = pd.Series({
            'trades': len(trades),
            'wins': wins,
            'win_rate': wins / len(trades),
            'total_points': trades['pnl_points'].sum(),
            'avg_points': trades['pnl_points'].mean(),
            'median_points': trades['pnl_points'].median()
        })
    return trades, summary
# You already built spot_1min (09:15–15:25) as shown in your snippet
daily = make_daily_from_1min(spot_1min)

feats = compute_nr7_and_stretch(daily, noise_win=10)   # NR7 flags + Stretch (shifted)

trades, summary = backtest_nr7_orb_open_stretch(
    spot_1min, feats,
    k=1.25, market_open="09:15", market_close="15:25",
    target_points=200
)

print(summary)        # trades, wins, win_rate, total/avg/median points
print(trades.head())  # includes Stretch, weekday, hit_50/75/100/150/200, etc.

/tmp/ipykernel_315450/3583682481.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d['NR7_prev'] = d['NR7'].shift(1).fillna(False)


trades            131.000000
wins               47.000000
win_rate            0.358779
total_points     2259.006250
avg_points         17.244323
median_points       0.000000
dtype: float64
        Date    weekday  NR7_prev  Open_0915  Stretch  k_used     buy_stop  \
0 2021-06-16  Wednesday      True   15848.90   32.005    1.25  15888.90625   
1 2021-06-25     Friday      True   15844.45   30.250    1.25  15882.26250   
2 2021-07-02     Friday      True   15711.20   24.005    1.25  15741.20625   
3 2021-07-06    Tuesday      True   15821.85   27.410    1.25  15856.11250   
4 2021-07-14  Wednesday      True   15811.65   28.655    1.25  15847.46875   

     sell_stop   side          entry_time  ...           exit_time  \
0  15808.89375  SHORT 2021-06-16 10:21:00  ... 2021-06-16 13:57:00   
1  15806.63750  SHORT 2021-06-25 09:21:00  ... 2021-06-25 10:32:00   
2  15681.19375  SHORT 2021-07-02 09:20:00  ... 2021-07-02 10:27:00   
3  15787.58750   LONG 2021-07-06 09:18:00  ... 2021-07-06 10:1